## Observações
- A maioria das associações individuais é fraca: o V de Cramér mediano foi aproximadamente 0,080, enquanto 95% dos loci apresentaram valores inferiores a 0,178.
- Aproximadamente 70,6% dos loci produziram tabelas de contingência consideradas esparsas. Além disso, a mediana da menor contagem esperada foi 3,07, abaixo da referência convencional de 5, indicando que a aproximação do teste qui-quadrado pode ser imprecisa para muitos loci.
- Embora 404 loci apresentem `p_value < 0,05` e 142 apresentem `p_value < 0,01`, apenas 3 loci permanecem significativos após a correção para múltiplos testes, considerando `q_value < 0,05`.
- `chr1_16759007`, `chr9_39118208` e `chr4_108014614` foram considerados significativos após controle da FDR (False Discovery Rate).
- O locus `chr1_16759007` apresentou a evidência estatística mais forte e a maior magnitude de associação. Entretanto, sua tabela de contingência é esparsa, portanto o resultado do qui-quadrado deve ser interpretado com cautela. Pode exigir testes adicionais.
- Foram encontradas correlações elevadas entre alguns pares de loci, chegando a aproximadamente 0,97.
- As maiores correlações ocorrem principalmente entre loci fisicamente muito próximos, com destaque para um grupo na região `chr6_299441xx`. Isso sugere redundância local possivelmente decorrente de desequilíbrio de ligação.

## Decisões

- Treinar um modelo interpretável utilizando somente os 3 locis mais significativos após `q_value < 0,05`.
- Treinar um modelo interpretável utilizando somente os 14 locis mais significativos após `q_value < 0,10`.
- Manter um representante de cada grupo de loci altamente correlacionados, considerando um limiar de correlação de 0,95.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from covid.constants import INTERIM_TRAIN_DATA_PATH

train_data = pd.read_csv(INTERIM_TRAIN_DATA_PATH, dtype={"id": str})
train_data.shape

In [ ]:
from covid.feature import TARGET, get_loci_data
from covid.eda.bivariate import association_summary, loci_correlation_from_summary, strongest_pairs

association_summary = association_summary(
    loci_data=get_loci_data(train_data), target=train_data[TARGET]
)
association_summary

In [ ]:
percentiles = [0.01, 0.05, 0.1, 0.5, 0.75, 0.95, 0.99]
association_summary.describe(percentiles=percentiles).T

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 5))

axes[0].hist(association_summary["q_value"], bins=50)
axes[0].set_title("Distribution of q-values for association between loci and target")
axes[0].set_xlabel("q-value")
axes[0].set_ylabel("Number of loci")

axes[1].hist(association_summary["cramers_v"], bins=50)
axes[1].set_title("Distribution of Cramér's V for association between loci and target")
axes[1].set_xlabel("Cramér's V")
axes[1].set_ylabel("Number of loci")

plt.tight_layout()
plt.show()

In [ ]:
pd.Series({
    "p < 0.05": association_summary["p_value"].lt(0.05).sum(),
    "p < 0.01": association_summary["p_value"].lt(0.01).sum(),
    "q < 0.05": association_summary["q_value"].lt(0.05).sum(),
    "q < 0.10": association_summary["q_value"].lt(0.10).sum(),
})

In [ ]:
associated = (
    association_summary[["q_value", "cramers_v", "sparse_table"]]
    .query("q_value < 0.05")
    .sort_values("cramers_v", ascending=False)
)
associated

In [ ]:
loci_correlations = loci_correlation_from_summary(
    association_summary_data=association_summary,
    loci_data=get_loci_data(train_data),
    max_loci=50
)
loci_correlations.iloc[:4, :4]

In [ ]:
import seaborn as sns
import numpy as np

upper_mask = np.triu(np.ones_like(loci_correlations, dtype=bool), k=1)
sns.clustermap(
    loci_correlations.fillna(0),
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    figsize=(16, 16),
    linewidths=0.1,
    xticklabels=True,
    yticklabels=True,
    cbar_kws={"label": "Spearman correlation"},
)

In [ ]:
strongest_pairs(
    loci_correlations=loci_correlations,
    correlation_threshold=0.8
)